In [ ]:
import numpy as np
import scipy
import scipy.sparse as sp
print("NumPy :", np.__version__)
print("SciPy :", scipy.__version__)

if not hasattr(sp.dok_matrix, "_update"):

    def _dok_update(self, data):
        if hasattr(data, "keys"):
            for key in data.keys():
                self[key] = data[key]
        else:
            for key, value in data:
                self[key] = value

    sp.dok_matrix._update = _dok_update

_NUMPY_REMOVED_ALIASES = {
    "float_": np.float64,
    "complex_": np.complex128,
    "int_": np.int64,
    "longfloat": np.longdouble,
    "singlecomplex": np.complex64,
    "cfloat": np.complex128,
    "clongfloat": np.clongdouble,
    "string_": np.bytes_,
    "unicode_": np.str_,
    "object0": np.object_,
    "bytes0": np.bytes_,
    "str0": np.str_,
    "int0": np.intp,
    "uint0": np.uintp,
    "void0": np.void,
}

for alias, replacement in _NUMPY_REMOVED_ALIASES.items():
    if not hasattr(np, alias):
        setattr(np, alias, replacement)

import os
import ast
import time
import gc
import traceback
import logging
import sys

import pandas as pd
import torch
import kagglehub

from sklearn.model_selection import train_test_split

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.model.context_aware_recommender.deepfm import DeepFM
from recbole.model.context_aware_recommender.widedeep import WideDeep
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from recbole.utils.case_study import full_sort_topk

from surprise import SVD, Dataset as SurpriseDataset, Reader


print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "CUDA device:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
!pip install -q recbole surprise kagglehub kmeans_pytorch

In [ ]:
#logging
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)

logging.getLogger("recbole").setLevel(logging.INFO)

In [ ]:
#datasets
mlpath = kagglehub.dataset_download(
    "sherinclaudia/movielens"
)

print("MovieLens path:", mlpath)

krpath = kagglehub.dataset_download(
    "arashnic/kuairec-recommendation-system-data-density-100"
)

print("KuaiRec path:", krpath)

ML_BASE = "/kaggle/input/movielens"
KR_BASE = (
    "/kaggle/input/"
    "kuairec-recommendation-system-data-density-100/"
    "KuaiRec 2.0/data"
)

if not os.path.exists(ML_BASE):

    possible_ml = [
        mlpath,
        "/kaggle/input/movielens",
    ]

    for candidate in possible_ml:
        if os.path.exists(
            os.path.join(candidate, "movies.dat")
        ):
            ML_BASE = candidate
            break

if not os.path.exists(KR_BASE):

    possible_kr = [
        krpath,
        os.path.join(krpath, "KuaiRec 2.0", "data"),
        "/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data",
    ]

    for candidate in possible_kr:

        if os.path.exists(
            os.path.join(candidate, "big_matrix.csv")
        ):
            KR_BASE = candidate
            break


print("MovieLens directory:", ML_BASE)
print("KuaiRec directory:", KR_BASE)

In [ ]:
#loading ml-1m
ml_movies = pd.read_csv(
    os.path.join(ML_BASE, "movies.dat"),
    sep="::",
    names=[
        "MovieID",
        "Title",
        "Genres",
    ],
    engine="python",
    encoding="latin-1",
)

ml_movies["Genres"] = (
    ml_movies["Genres"]
    .fillna("")
    .str.split("|")
    .apply(
        lambda x: [
            g for g in x
            if g
        ]
    )
)


ml_ratings = pd.read_csv(
    os.path.join(ML_BASE, "ratings.dat"),
    sep="::",
    names=[
        "UID",
        "MovieID",
        "Rating",
        "Timestamp",
    ],
    engine="python",
    encoding="latin-1",
)


ml_users = pd.read_csv(
    os.path.join(ML_BASE, "users.dat"),
    sep="::",
    names=[
        "UID",
        "SEX",
        "AGE",
        "OCC",
        "PIN",
    ],
    engine="python",
    encoding="latin-1",
)


print("Movies :", ml_movies.shape)
print("Ratings:", ml_ratings.shape)
print("Users  :", ml_users.shape)

In [ ]:
#loading kr
kr_bigm = pd.read_csv(
    os.path.join(
        KR_BASE,
        "big_matrix.csv"
    )
)

kr_smallm = pd.read_csv(
    os.path.join(
        KR_BASE,
        "small_matrix.csv"
    )
)

kr_cats = pd.read_csv(
    os.path.join(
        KR_BASE,
        "item_categories.csv"
    )
)


print("KuaiRec big matrix  :", kr_bigm.shape)
print("KuaiRec small matrix:", kr_smallm.shape)
print("KuaiRec categories  :", kr_cats.shape)

In [ ]:
assert set(
    kr_smallm.user_id
).issubset(
    set(kr_bigm.user_id)
)

assert set(
    kr_smallm.video_id
).issubset(
    set(kr_bigm.video_id)
)

print("KuaiRec: small_matrix fully covered by big_matrix vocabulary.")

In [ ]:
KR_THRESHOLD = 2.0
ML_THRESHOLD = 3.5


kr_bigm["label"] = (
    kr_bigm["watch_ratio"] > KR_THRESHOLD
).astype(int)


kr_smallm["label"] = (
    kr_smallm["watch_ratio"] > KR_THRESHOLD
).astype(int)


ml_ratings["label"] = (
    ml_ratings["Rating"] > ML_THRESHOLD
).astype(int)


print("\nPositive rates:")
print(
    "KuaiRec big  :",
    kr_bigm["label"].mean()
)

print(
    "KuaiRec small:",
    kr_smallm["label"].mean()
)

print(
    "MovieLens    :",
    ml_ratings["label"].mean()
)

In [ ]:
kr_user2idx = {
    u: i
    for i, u in enumerate(
        sorted(
            kr_bigm.user_id.unique()
        )
    )
}

kr_item2idx = {
    v: i
    for i, v in enumerate(
        sorted(
            kr_bigm.video_id.unique()
        )
    )
}


for df in [
    kr_bigm,
    kr_smallm
]:

    df["user_idx"] = (
        df["user_id"]
        .map(kr_user2idx)
        .astype(int)
    )

    df["item_idx"] = (
        df["video_id"]
        .map(kr_item2idx)
        .astype(int)
    )


n_kr_items = len(kr_item2idx)

ml_user2idx = {
    u: i
    for i, u in enumerate(
        sorted(
            ml_ratings.UID.unique()
        )
    )
}

ml_item2idx = {
    m: i
    for i, m in enumerate(
        sorted(
            ml_ratings.MovieID.unique()
        )
    )
}


ml_ratings["user_idx"] = (
    ml_ratings["UID"]
    .map(ml_user2idx)
    .astype(int)
)

ml_ratings["item_idx"] = (
    ml_ratings["MovieID"]
    .map(ml_item2idx)
    .astype(int)
)


n_ml_items = len(ml_item2idx)


print(
    "MovieLens users/items:",
    len(ml_user2idx),
    n_ml_items
)

print(
    "KuaiRec users/items:",
    len(kr_user2idx),
    n_kr_items
)

In [ ]:
ml_train, ml_test = train_test_split(
    ml_ratings,
    test_size=0.20,
    random_state=42,
    stratify=ml_ratings["label"],
)

print(
    f"\nMovieLens: "
    f"{len(ml_train)} train / "
    f"{len(ml_test)} test interactions"
)

In [ ]:
kr_train_pos_pairs = set(
    zip(
        kr_bigm.loc[
            kr_bigm["label"] == 1,
            "user_idx"
        ],
        kr_bigm.loc[
            kr_bigm["label"] == 1,
            "item_idx"
        ],
    )
)


kr_eval_pos_pairs = set(
    zip(
        kr_smallm.loc[
            kr_smallm["label"] == 1,
            "user_idx"
        ],
        kr_smallm.loc[
            kr_smallm["label"] == 1,
            "item_idx"
        ],
    )
)


overlap_pairs = (
    kr_train_pos_pairs
    &
    kr_eval_pos_pairs
)


print(
    f"\nKuaiRec overlapping positive pairs: "
    f"{len(overlap_pairs)} / "
    f"{len(kr_eval_pos_pairs)}"
)


kr_smallm["pair"] = list(
    zip(
        kr_smallm["user_idx"],
        kr_smallm["item_idx"]
    )
)


kr_smallm_clean = (
    kr_smallm[
        ~kr_smallm["pair"].isin(
            overlap_pairs
        )
    ]
    .drop(columns=["pair"])
    .reset_index(drop=True)
)


print(
    f"KuaiRec test rows: "
    f"{len(kr_smallm)} -> "
    f"{len(kr_smallm_clean)} after leakage masking"
)

In [ ]:
BASE = "/content/recbole_data"

DATASET_DIRS = [
    "ml-1m",
    "kuairec",
]

for name in DATASET_DIRS:

    os.makedirs(
        os.path.join(BASE, name),
        exist_ok=True
    )

In [ ]:
def write_inter_file(
    df,
    path,
    user_col="user_idx",
    item_col="item_idx",
    label_col="label",
):

    out = pd.DataFrame(
        {
            "user_id:token":
                df[user_col]
                .astype(str),

            "item_id:token":
                df[item_col]
                .astype(str),

            "label:float":
                df[label_col]
                .astype(float),
        }
    )

    out.to_csv(
        path,
        sep="\t",
        index=False
    )


def write_item_file_multivalued(
    item_ids,
    feat_lists,
    path,
    feat_field="genre",
):

    rows = []

    for iid, feats in zip(
        item_ids,
        feat_lists
    ):

        if feats is None:
            feats = []

        feats = [
            str(x)
            for x in feats
            if str(x).strip()
        ]

        feat_str = " ".join(feats)

        rows.append(
            (
                str(int(iid)),
                feat_str
            )
        )

    out = pd.DataFrame(
        rows,
        columns=[
            "item_id:token",
            f"{feat_field}:token_seq"
        ]
    )

    out.to_csv(
        path,
        sep="\t",
        index=False
    )

In [ ]:
ml_valid = ml_train.sample(
    frac=0.10,
    random_state=42
)


ml_train_final = ml_train.drop(
    ml_valid.index
)


ML_DIR = os.path.join(
    BASE,
    "ml-1m"
)


write_inter_file(
    ml_train_final,
    os.path.join(
        ML_DIR,
        "ml-1m.train.inter"
    )
)


write_inter_file(
    ml_valid,
    os.path.join(
        ML_DIR,
        "ml-1m.valid.inter"
    )
)


write_inter_file(
    ml_test,
    os.path.join(
        ML_DIR,
        "ml-1m.test.inter"
    )
)


ml_movie_mask = (
    ml_movies["MovieID"]
    .map(ml_item2idx)
    .notna()
)


ml_movie_item_ids = (
    ml_movies.loc[
        ml_movie_mask,
        "MovieID"
    ]
    .map(ml_item2idx)
    .astype(int)
    .tolist()
)


ml_movie_genres = (
    ml_movies.loc[
        ml_movie_mask,
        "Genres"
    ]
    .tolist()
)


write_item_file_multivalued(
    ml_movie_item_ids,
    ml_movie_genres,
    os.path.join(
        ML_DIR,
        "ml-1m.item"
    ),
    feat_field="genre"
)


print(
    "MovieLens items written:",
    len(ml_movie_item_ids),
    "/",
    len(ml_movies)
)

In [ ]:
kr_valid = kr_bigm.sample(
    frac=0.02,
    random_state=42
)


kr_train_final = kr_bigm.drop(
    kr_valid.index
)


KR_DIR = os.path.join(
    BASE,
    "kuairec"
)


write_inter_file(
    kr_train_final,
    os.path.join(
        KR_DIR,
        "kuairec.train.inter"
    )
)


write_inter_file(
    kr_valid,
    os.path.join(
        KR_DIR,
        "kuairec.valid.inter"
    )
)


write_inter_file(
    kr_smallm_clean,
    os.path.join(
        KR_DIR,
        "kuairec.test.inter"
    )
)

kr_cats["feat"] = (
    kr_cats["feat"]
    .apply(
        lambda x:
        ast.literal_eval(x)
        if isinstance(x, str)
        else x
    )
)


kr_items_mask = (
    kr_cats["video_id"]
    .map(kr_item2idx)
    .notna()
)


kr_item_ids = (
    kr_cats.loc[
        kr_items_mask,
        "video_id"
    ]
    .map(kr_item2idx)
    .astype(int)
    .tolist()
)


kr_item_feats = (
    kr_cats.loc[
        kr_items_mask,
        "feat"
    ]
    .tolist()
)


write_item_file_multivalued(
    kr_item_ids,
    kr_item_feats,
    os.path.join(
        KR_DIR,
        "kuairec.item"
    ),
    feat_field="feat"
)


print(
    "KuaiRec items written:",
    len(kr_item_ids),
    "/",
    len(kr_cats)
)


print("\nAtomic files written successfully.")

In [ ]:
print("\nMovieLens .item:")
print(
    pd.read_csv(
        os.path.join(
            ML_DIR,
            "ml-1m.item"
        ),
        sep="\t"
    ).head()
)


print("\nKuaiRec .item:")
print(
    pd.read_csv(
        os.path.join(
            KR_DIR,
            "kuairec.item"
        ),
        sep="\t"
    ).head()
)

In [ ]:
#metrics
def compute_recall_ndcg_at_k(
    user_topk,
    user_ground_truth,
    k=20,
):

    recalls = []
    ndcgs = []

    for user, ranked_items in user_topk.items():

        gt = user_ground_truth.get(
            user,
            set()
        )

        if not gt:
            continue

        ranked_k = ranked_items[:k]

        hits = np.array(
            [
                1 if item in gt else 0
                for item in ranked_k
            ],
            dtype=float
        )

        recall = (
            hits.sum()
            /
            min(len(gt), k)
        )

        recalls.append(
            recall
        )

        discounts = 1.0 / np.log2(
            np.arange(
                2,
                len(ranked_k) + 2
            )
        )

        dcg = np.sum(
            hits * discounts
        )

        ideal_hits = min(
            len(gt),
            k
        )

        ideal_discounts = (
            1.0
            /
            np.log2(
                np.arange(
                    2,
                    ideal_hits + 2
                )
            )
        )

        idcg = np.sum(
            ideal_discounts
        )

        ndcg = (
            dcg / idcg
            if idcg > 0
            else 0.0
        )

        ndcgs.append(
            ndcg
        )

    if not recalls:

        return {
            f"Recall@{k}": 0.0,
            f"NDCG@{k}": 0.0,
        }

    return {
        f"Recall@{k}":
            float(np.mean(recalls)),

        f"NDCG@{k}":
            float(np.mean(ndcgs)),
    }

In [ ]:
#result management
RESULTS_PATH = (
    "/content/results_log.csv"
)


RESULT_SCHEMA = [
    "dataset",
    "model",
    "seed",
    "Recall@20",
    "NDCG@20",
    "error",
]


def load_completed_runs():

    if not os.path.exists(
        RESULTS_PATH
    ):
        return set()

    df = pd.read_csv(
        RESULTS_PATH
    )

    if "error" not in df.columns:
        return set()

    successful = df[
        df["error"].isna()
    ]

    return set(
        zip(
            successful["dataset"],
            successful["model"],
            successful["seed"],
        )
    )


def append_result(row):

    full_row = {
        col: row.get(
            col,
            np.nan
        )
        for col in RESULT_SCHEMA
    }

    df = pd.DataFrame(
        [full_row],
        columns=RESULT_SCHEMA
    )

    header = not os.path.exists(
        RESULTS_PATH
    )

    df.to_csv(
        RESULTS_PATH,
        mode="a",
        header=header,
        index=False
    )


def clean_failed_rows():

    if not os.path.exists(
        RESULTS_PATH
    ):
        return

    df = pd.read_csv(
        RESULTS_PATH
    )

    for col in RESULT_SCHEMA:

        if col not in df.columns:
            df[col] = np.nan

    df = df[
        RESULT_SCHEMA
    ]

    success = df[
        df["error"].isna()
    ]

    failed = (
        df[
            df["error"].notna()
        ]
        .drop_duplicates(
            subset=[
                "dataset",
                "model",
                "seed"
            ],
            keep="last"
        )
    )

    success_keys = set(
        zip(
            success["dataset"],
            success["model"],
            success["seed"],
        )
    )

    failed = failed[
        ~failed.apply(
            lambda r:
            (
                r["dataset"],
                r["model"],
                r["seed"],
            )
            in success_keys,
            axis=1
        )
    ]

    out = pd.concat(
        [
            success,
            failed
        ],
        ignore_index=True
    )

    out.to_csv(
        RESULTS_PATH,
        index=False
    )

    print(
        f"Cleaned results: "
        f"{len(success)} successes, "
        f"{len(failed)} pending failures"
    )


clean_failed_rows()

In [ ]:
#recbole config
def build_recbole_config(
    dataset_name,
    model_name,
    use_content,
    emb_dim=64,
    seed=0,
):

    load_col = {
        "inter": [
            "user_id",
            "item_id",
            "label",
        ]
    }


    if use_content:

        feat_field = (
            "genre"
            if dataset_name == "ml-1m"
            else "feat"
        )

        load_col["item"] = [
            "item_id",
            feat_field,
        ]

    config_dict = {

        "data_path":
            "/content/recbole_data",

        "dataset":
            dataset_name,

        "load_col":
            load_col,

        "USER_ID_FIELD":
            "user_id",

        "ITEM_ID_FIELD":
            "item_id",

        "LABEL_FIELD":
            "label",

        "field_separator":
            "\t",

        "embedding_size":
            emb_dim,

        "mlp_hidden_size":
            [128, 64],

        "epochs":
            3,

        "train_batch_size":
            4096,

        "eval_batch_size":
            8192,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-6,

        "seed":
            seed,

        "reproducibility":
            True,

        "benchmark_filename":
            [
                "train",
                "valid",
                "test",
            ],

        "eval_args": {

            "split": {
                "RS": [
                    0,
                    0,
                    0
                ]
            },

            "group_by":
                "user",

            "order":
                "RO",

            "mode": {
                "valid":
                    "full",

                "test":
                    "full",
            },
        },

        "metrics":
            [
                "Recall",
                "NDCG"
            ],

        "topk":
            [20],

        "valid_metric":
            "NDCG@20",

        "stopping_step":
            3,

        "eval_step":
            999999,

        "save_step":
            999999,
    }


    # LightGCN negative sampling


    if model_name == "LightGCN":

        config_dict[
            "train_neg_sample_args"
        ] = {

            "distribution":
                "uniform",

            "sample_num":
                1,

            "alpha":
                1.0,

            "dynamic":
                False,

            "candidate_num":
                0,
        }


    # Content features

    if use_content:

        feat_field = (
            "genre"
            if dataset_name == "ml-1m"
            else "feat"
        )

        config_dict[
            "selected_features"
        ] = [
            feat_field
        ]


    model_cls = {

        "DeepFM":
            DeepFM,

        "WideDeep":
            WideDeep,

        "LightGCN":
            LightGCN,

    }[model_name]


    config = Config(
        model=model_cls,
        dataset=dataset_name,
        config_dict=config_dict,
    )


    return config, model_cls

In [ ]:
#training recbole
def train_recbole_model(
    dataset_name,
    model_name,
    use_content,
    seed,
    emb_dim=64,
):

    config, model_cls = (
        build_recbole_config(
            dataset_name,
            model_name,
            use_content,
            emb_dim,
            seed,
        )
    )


    # Reproducibility

    init_seed(
        config["seed"],
        config["reproducibility"]
    )


    init_logger(
        config
    )


    # Dataset

    print(
        "  Creating RecBole dataset...",
        flush=True
    )

    dataset = create_dataset(
        config
    )


    print(
        f"  Dataset created: "
        f"{dataset.user_num} users, "
        f"{dataset.item_num} items, "
        f"{len(dataset.inter_feat)} interactions",
        flush=True
    )

    print(
        "  Preparing data loaders...",
        flush=True
    )

    (
        train_data,
        valid_data,
        test_data
    ) = data_preparation(
        config,
        dataset
    )


    print(
        f"  Loaders ready: "
        f"train={len(train_data)}, "
        f"valid={len(valid_data)}, "
        f"test={len(test_data)}",
        flush=True
    )


    model = model_cls(
        config,
        train_data.dataset
    ).to(
        config["device"]
    )


    # Trainer

    trainer = Trainer(
        config,
        model
    )


    # Training

    print(
        "  Starting training...",
        flush=True
    )


    trainer.fit(
        train_data,
        valid_data,
        verbose=False,
        show_progress=True
    )


    print(
        "  Training complete.",
        flush=True
    )


    return (
        model,
        trainer,
        dataset,
        train_data,
        valid_data,
        test_data,
        config,
    )

In [ ]:
def recbole_token_to_id(
    dataset,
    field,
    token,
):

    token = str(token)

    try:

        result = dataset.token2id(
            field,
            [token]
        )

        return int(
            result[0]
        )

    except Exception:

        return None

In [ ]:
def get_ground_truth_recbole(
    dataset_name,
    dataset,
    split="test",
):

    path = (
        f"/content/recbole_data/"
        f"{dataset_name}/"
        f"{dataset_name}.{split}.inter"
    )


    df = pd.read_csv(
        path,
        sep="\t"
    )


    df.columns = [
        "user_id",
        "item_id",
        "label",
    ]


    positives = df[
        df["label"] == 1.0
    ]


    ground_truth = {}


    for row in positives.itertuples(
        index=False
    ):

        uid = recbole_token_to_id(
            dataset,
            dataset.uid_field,
            row.user_id
        )

        iid = recbole_token_to_id(
            dataset,
            dataset.iid_field,
            row.item_id
        )


        if uid is None or iid is None:
            continue


        ground_truth.setdefault(
            uid,
            set()
        ).add(
            iid
        )


    return ground_truth

In [ ]:
def get_recbole_topk_predictions(
    model,
    dataset,
    test_data,
    config,
    eval_users,
    k=20,
    batch_size=256,
):

    model.eval()

    device = config["device"]

    user_topk = {}

    eval_users = np.asarray(
        eval_users,
        dtype=np.int64
    )

    with torch.no_grad():

        for start in range(
            0,
            len(eval_users),
            batch_size
        ):

            batch_users = (
                eval_users[
                    start:
                    start + batch_size
                ]
            )


            print(
                f"    Evaluating users "
                f"{start + 1}-"
                f"{min(start + batch_size, len(eval_users))}"
                f"/{len(eval_users)}",
                end="\r",
                flush=True
            )

            topk_scores, topk_items = (
                full_sort_topk(
                    batch_users,
                    model,
                    test_data,
                    k,
                    device,
                )
            )


            topk_items = (
                topk_items
                .detach()
                .cpu()
                .numpy()
            )


            for row_idx, uid in enumerate(
                batch_users
            ):

                user_topk[
                    int(uid)
                ] = (
                    topk_items[row_idx]
                    .astype(int)
                    .tolist()
                )


    print(
        "\n",
        flush=True
    )


    return user_topk

In [ ]:
def train_surprise_svd(
    train_df,
    n_factors=64,
    seed=0,
):

    reader = Reader(
        rating_scale=(0, 1)
    )


    data = (
        SurpriseDataset
        .load_from_df(
            train_df[
                [
                    "user_idx",
                    "item_idx",
                    "label"
                ]
            ],
            reader
        )
    )


    trainset = (
        data.build_full_trainset()
    )


    algo = SVD(
        n_factors=n_factors,
        random_state=seed
    )


    algo.fit(
        trainset
    )


    return algo

In [ ]:
def get_surprise_topk_predictions(
    algo,
    train_user_pos,
    all_item_ids,
    eval_users,
    k=20,
):

    user_topk = {}


    for idx, u in enumerate(
        eval_users
    ):

        if idx % 25 == 0:

            print(
                f"    SVD scoring "
                f"{idx}/{len(eval_users)}",
                end="\r",
                flush=True
            )


        seen = train_user_pos.get(
            u,
            set()
        )


        candidates = [
            i
            for i in all_item_ids
            if i not in seen
        ]


        scores = [
            (
                i,
                algo.predict(
                    int(u),
                    int(i)
                ).est
            )
            for i in candidates
        ]


        scores.sort(
            key=lambda x: -x[1]
        )


        user_topk[
            int(u)
        ] = [
            int(i)
            for i, _ in scores[:k]
        ]


    print(
        "\n",
        flush=True
    )


    return user_topk

In [ ]:
DATASETS = [
    "ml-1m",
    "kuairec",
]


CONFIGS = [

    ("DeepFM", True),

    ("DeepFM", False),

    ("WideDeep", True),

    ("WideDeep", False),

    ("LightGCN", None),

    ("SVD", None),

]


SEEDS = [
    0,
    1,
    2,
    3,
    4,
]


def run_name(
    model_name,
    use_content
):

    if use_content is None:
        return model_name

    return (
        f"{model_name}_"
        f"{'on' if use_content else 'off'}"
    )

In [ ]:
def run_all(
    datasets=DATASETS,
    configs=CONFIGS,
    seeds=SEEDS,
    k=20,
    verbose=True,
):

    completed = (
        load_completed_runs()
    )


    total_runs = (
        len(datasets)
        *
        len(configs)
        *
        len(seeds)
    )


    run_counter = 0


    for dataset_name in datasets:

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"DATASET: {dataset_name}"
        )

        print(
            "=" * 70,
            flush=True
        )


        for model_name, use_content in configs:

            name = run_name(
                model_name,
                use_content
            )


            for seed in seeds:

                run_counter += 1


                progress_tag = (
                    f"[{run_counter}/{total_runs}]"
                )


                key = (
                    dataset_name,
                    name,
                    seed
                )


                # ------------------------------------------------
                # Skip successful runs
                # ------------------------------------------------

                if key in completed:

                    print(
                        f"{progress_tag} "
                        f"SKIP: "
                        f"{dataset_name} / "
                        f"{name} / "
                        f"seed={seed}",
                        flush=True
                    )

                    continue


                print(
                    "\n"
                    + "-" * 70
                )

                print(
                    f"{progress_tag} "
                    f"START: "
                    f"{dataset_name} / "
                    f"{name} / "
                    f"seed={seed}"
                )

                print(
                    "-" * 70,
                    flush=True
                )


                run_t0 = time.time()


                # ------------------------------------------------
                # Initialize objects
                # ------------------------------------------------

                model = None
                trainer = None
                ds = None
                train_data = None
                valid_data = None
                test_data = None
                config = None
                algo = None
                user_topk = None


                try:

                    # ==================================================
                    # SVD
                    # ==================================================

                    if model_name == "SVD":

                        train_path = (
                            f"/content/recbole_data/"
                            f"{dataset_name}/"
                            f"{dataset_name}.train.inter"
                        )


                        train_df = pd.read_csv(
                            train_path,
                            sep="\t"
                        )


                        train_df.columns = [
                            "user_idx",
                            "item_idx",
                            "label",
                        ]


                        train_df[
                            "user_idx"
                        ] = (
                            train_df[
                                "user_idx"
                            ]
                            .astype(int)
                        )


                        train_df[
                            "item_idx"
                        ] = (
                            train_df[
                                "item_idx"
                            ]
                            .astype(int)
                        )


                        train_df[
                            "label"
                        ] = (
                            train_df[
                                "label"
                            ]
                            .astype(float)
                        )


                        # ------------------------------------------
                        # Training positive interactions
                        # ------------------------------------------

                        train_user_pos = (

                            train_df[
                                train_df[
                                    "label"
                                ] == 1.0
                            ]

                            .groupby(
                                "user_idx"
                            )[
                                "item_idx"
                            ]

                            .apply(set)

                            .to_dict()
                        )


                        all_item_ids = sorted(
                            train_df[
                                "item_idx"
                            ]
                            .unique()
                            .tolist()
                        )


                        # ------------------------------------------
                        # Ground truth
                        # ------------------------------------------

                        test_path = (
                            f"/content/recbole_data/"
                            f"{dataset_name}/"
                            f"{dataset_name}.test.inter"
                        )


                        test_df = pd.read_csv(
                            test_path,
                            sep="\t"
                        )


                        test_df.columns = [
                            "user_idx",
                            "item_idx",
                            "label",
                        ]


                        positive_test = (
                            test_df[
                                test_df[
                                    "label"
                                ] == 1.0
                            ]
                        )


                        ground_truth = (

                            positive_test

                            .groupby(
                                "user_idx"
                            )[
                                "item_idx"
                            ]

                            .apply(set)

                            .to_dict()
                        )


                        eval_users = list(
                            ground_truth.keys()
                        )


                        # ------------------------------------------
                        # Train SVD
                        # ------------------------------------------

                        print(
                            "  Training Surprise SVD...",
                            flush=True
                        )


                        t0 = time.time()


                        algo = train_surprise_svd(
                            train_df,
                            n_factors=64,
                            seed=seed
                        )


                        print(
                            f"  SVD trained in "
                            f"{time.time() - t0:.1f}s",
                            flush=True
                        )


                        # ------------------------------------------
                        # Score
                        # ------------------------------------------

                        print(
                            f"  Scoring "
                            f"{len(eval_users)} users...",
                            flush=True
                        )


                        t0 = time.time()


                        user_topk = (
                            get_surprise_topk_predictions(
                                algo,
                                train_user_pos,
                                all_item_ids,
                                eval_users,
                                k=k
                            )
                        )


                        print(
                            f"  SVD scoring completed in "
                            f"{time.time() - t0:.1f}s",
                            flush=True
                        )


                    # ==================================================
                    # RECBole MODELS
                    # ==================================================

                    else:

                        print(
                            "  Training RecBole model...",
                            flush=True
                        )


                        t0 = time.time()


                        (
                            model,
                            trainer,
                            ds,
                            train_data,
                            valid_data,
                            test_data,
                            config,
                        ) = train_recbole_model(

                            dataset_name,

                            model_name,

                            use_content=(
                                bool(use_content)
                                if use_content is not None
                                else False
                            ),

                            seed=seed,
                        )


                        print(
                            f"  RecBole training completed in "
                            f"{time.time() - t0:.1f}s",
                            flush=True
                        )


                        # ------------------------------------------
                        # Ground truth
                        # ------------------------------------------

                        ground_truth = (
                            get_ground_truth_recbole(
                                dataset_name,
                                ds,
                                split="test"
                            )
                        )


                        eval_users = list(
                            ground_truth.keys()
                        )


                        print(
                            f"  Test users with positive "
                            f"ground truth: "
                            f"{len(eval_users)}",
                            flush=True
                        )


                        # ------------------------------------------
                        # Full-sort evaluation
                        # ------------------------------------------

                        print(
                            f"  Generating top-{k} "
                            f"recommendations...",
                            flush=True
                        )


                        t0 = time.time()


                        user_topk = (
                            get_recbole_topk_predictions(
                                model=model,
                                dataset=ds,
                                test_data=test_data,
                                config=config,
                                eval_users=eval_users,
                                k=k,
                                batch_size=256,
                            )
                        )


                        print(
                            f"  Full-sort evaluation completed in "
                            f"{time.time() - t0:.1f}s",
                            flush=True
                        )


                    # ==================================================
                    # METRICS
                    # ==================================================

                    metrics = (
                        compute_recall_ndcg_at_k(
                            user_topk,
                            ground_truth,
                            k=k
                        )
                    )


                    row = {
                        "dataset":
                            dataset_name,

                        "model":
                            name,

                        "seed":
                            seed,

                        **metrics,
                    }


                    append_result(
                        row
                    )


                    print(
                        "\n  RESULT:"
                    )

                    print(
                        f"    Recall@{k}: "
                        f"{metrics[f'Recall@{k}']:.6f}"
                    )

                    print(
                        f"    NDCG@{k}:   "
                        f"{metrics[f'NDCG@{k}']:.6f}"
                    )

                    print(
                        f"    Total time: "
                        f"{time.time() - run_t0:.1f}s",
                        flush=True
                    )


                except Exception as e:

                    print(
                        "\n"
                        + "!" * 70
                    )

                    print(
                        f"FAILED: "
                        f"{dataset_name} / "
                        f"{name} / "
                        f"seed={seed}"
                    )

                    print(
                        f"Error: {repr(e)}"
                    )

                    print(
                        "!" * 70,
                        flush=True
                    )


                    traceback.print_exc()


                    append_result({

                        "dataset":
                            dataset_name,

                        "model":
                            name,

                        "seed":
                            seed,

                        "Recall@20":
                            np.nan,

                        "NDCG@20":
                            np.nan,

                        "error":
                            repr(e),
                    })


                finally:

                    # ------------------------------------------------
                    # Cleanup
                    # ------------------------------------------------

                    model = None
                    trainer = None
                    ds = None
                    train_data = None
                    valid_data = None
                    test_data = None
                    config = None
                    algo = None
                    user_topk = None


                    gc.collect()


                    if torch.cuda.is_available():

                        torch.cuda.empty_cache()


    print(
        "\n"
        + "=" * 70
    )

    print(
        f"FINISHED: "
        f"{total_runs} runs attempted"
    )

    print(
        f"Results saved to: "
        f"{RESULTS_PATH}"
    )

    print(
        "=" * 70,
        flush=True
    )

In [ ]:
run_all(
     datasets=["ml-1m"],
     configs=CONFIGS,
     seeds=[0, 1, 2, 3, 4],
     k=20,
     verbose=True
)

In [ ]:
"""
run_all(
     datasets=["ml-1m"],
     configs=CONFIGS,
     seeds=[0, 1, 2, 3, 4],
     k=20,
     verbose=True
)
"""

In [ ]:
if os.path.exists(
    RESULTS_PATH
):

    results = pd.read_csv(
        RESULTS_PATH
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "RESULTS"
    )

    print(
        "=" * 70
    )

    print(
        results.to_string(
            index=False
        )
    )

    successful = results[
        results["error"].isna()
    ].copy()


    if len(successful) > 0:

        print(
            "\n"
            + "=" * 70
        )

        print(
            "MEAN RESULTS BY DATASET / MODEL"
        )

        print(
            "=" * 70
        )


        summary = (
            successful

            .groupby(
                [
                    "dataset",
                    "model"
                ]
            )

            [
                [
                    "Recall@20",
                    "NDCG@20"
                ]
            ]

            .agg(
                [
                    "mean",
                    "std"
                ]
            )
        )


        print(
            summary
        )